In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, KFold, cross_val_score


session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
    }
)

def get_games_by_publisher_steam(publisher_name, max_pages=3):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    catalog = []

    for page in range(1, max_pages + 1):
        url = f"https://store.steampowered.com/search/?publisher={requests.utils.quote(publisher_name)}&page={page}"
        response = requests.get(url, headers=headers)

        if response.status_code == 429:
            print(
                "Rate limited (429)! Waiting 10 seconds before retrying..."
            )
            time.sleep(10)
            continue

        if response.status_code != 200:
            print(f"Error fetching page {page}: {response.status_code}")
            break

        soup = BeautifulSoup(response.text, "html.parser")
        game_rows = soup.find_all("a", class_="search_result_row")

        if not game_rows:
            break

        for row in game_rows:
            title_elem = row.find("span", class_="title")
            appid = row.get("data-ds-appid")

            if title_elem and appid:
                catalog.append(
                    {"title": title_elem.text.strip(), "steam_appid": str(appid)}
                )

        time.sleep(1.5)

    df = pd.DataFrame(catalog)
    if not df.empty:
        df = df[~df["steam_appid"].str.contains(",")].copy()
        df = df.drop_duplicates(subset=["steam_appid"]).reset_index(drop=True)
        df = df[
            ~df["title"].str.contains(
                "DLC|VR|Soundtrack|Pack|Expansion|Season Pass", case=False
            )
        ].reset_index(drop=True)

    return df


def fetch_single_game_data(appid):
    details_url = f"https://store.steampowered.com/api/appdetails?appids={appid}&filters=basic"

    try:
        res = session.get(details_url, timeout=5)
        if res.status_code == 200:
            json_data = res.json()
            if (
                json_data
                and str(appid) in json_data
                and json_data[str(appid)].get("success")
            ):
                app_data = json_data[str(appid)]["data"]
                app_type = app_data.get("type", "").lower()

                if app_type == "game":
                    reviews_url = f"https://store.steampowered.com/appreviews/{appid}?json=1&purchase_type=all"
                    rev_res = session.get(reviews_url, timeout=5)
                    total_reviews = 0

                    if rev_res.status_code == 200:
                        rev_data = rev_res.json()
                        if rev_data.get("success") == 1:
                            total_reviews = (
                                rev_data.get("query_summary", {}).get(
                                    "total_reviews", 0
                                )
                            )

                    return {
                        "steam_appid": str(appid),
                        "title": app_data.get("name", ""),
                        "type": app_type,
                        "total_reviews": total_reviews,
                    }
    except Exception as e:
        pass

    return None



def get_publisher_by_appid(appid):
    details_url = f"https://store.steampowered.com/api/appdetails?appids={appid}&filters=basic,publishers"
    try:
        res = session.get(details_url, timeout=5)
        if res.status_code == 200:
            json_data = res.json()
            if json_data and str(appid) in json_data and json_data[str(appid)].get("success"):
                publishers = json_data[str(appid)]["data"].get("publishers", [])
                if publishers:
                    return publishers[0]
    except Exception as e:
        pass
    return None




def get_verified_games_with_reviews_fast(appid_list, max_workers=6):
    valid_games = []
    print(
        f"Inspecting {len(appid_list)} potential AppIDs using {max_workers} parallel workers..."
    )

    start_time = time.time()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_appid = {
            executor.submit(fetch_single_game_data, appid): appid
            for appid in appid_list
        }

        for future in as_completed(future_to_appid):
            result = future.result()
            if result:
                valid_games.append(result)

    elapsed = time.time() - start_time
    print(f"Finished processing in {elapsed:.2f} seconds!")

    return pd.DataFrame(valid_games)

def filter_by_review_percentile(df_games, drop_percentile, min_catalog_size):
    
    if df_games.empty:
        return df_games

    total_games = len(df_games)

    if total_games <= min_catalog_size:
        print(
            f"\n--- Publisher Review Percentile Analysis ---\n"
            f"Catalog is small ({total_games} games <= {min_catalog_size}). Skipping percentile trim.\n"
        )
        return df_games.reset_index(drop=True)
        
    cutoff_threshold = df_games["total_reviews"].quantile(drop_percentile)

    print(f"\n--- Publisher Review Percentile Analysis ---")
    print(f"Total base games evaluated: {len(df_games)}")
    print(f"Calculated {int(drop_percentile * 100)}th percentile cutoff: {cutoff_threshold:.1f} reviews")

    df_games = df_games[df_games["total_reviews"] > 0].reset_index(drop=True)
    filtered_df = df_games[
        df_games["total_reviews"] > cutoff_threshold
    ].reset_index(drop=True)

    print(f"Remaining games after dropping bottom {int(drop_percentile * 100)}%: {len(filtered_df)}")
    return filtered_df

def publisher_catalog_dict(publisher, drop_percentile=0.10, min_catalog_size=15):
    raw_publisher_df = get_games_by_publisher_steam(
        publisher, max_pages=30
    )
    appid_list = raw_publisher_df["steam_appid"].tolist()
    
    catalog_df = get_verified_games_with_reviews_fast(appid_list, max_workers=6)
    
    final_catalog_df = filter_by_review_percentile(
        catalog_df, drop_percentile, min_catalog_size
    )
    
    final_catalog_dict=pd.Series(final_catalog_df["title"].values, index=final_catalog_df["steam_appid"]).to_dict()
    return final_catalog_dict




def get_game_price_history_itad(steam_appid, api_key, since_date="2015-01-01T00:00:00Z"):
    lookup_url = "https://api.isthereanydeal.com/games/lookup/v1"
    lookup_res = requests.get(
        lookup_url, params={"key": api_key, "appid": steam_appid}
    ).json()

    game_id = lookup_res.get("game", {}).get("id")
    if not game_id:
        print(f"Could not find ITAD game ID for Steam AppID {steam_appid}")
        return pd.DataFrame()

    history_url = "https://api.isthereanydeal.com/games/history/v2"
    params = {
        "key": api_key,
        "id": game_id,
        "shops": 61,
        "since": since_date
    }

    history_res = requests.get(history_url, params=params)

    if history_res.status_code != 200:
        print(f"Error fetching history: {history_res.status_code}")
        return pd.DataFrame()

    history_data = history_res.json()

    records = []
    for entry in history_data:
        deal = entry.get("deal", {})
        records.append(
            {
                "timestamp_raw": entry.get("timestamp"),
                "date": pd.to_datetime(entry.get("timestamp")),
                "price": deal.get("price", {}).get("amount"),
                "regular_price": deal.get("regular", {}).get("amount"),
                "cut": deal.get("cut", 0),
            }
        )

    df = pd.DataFrame(records)

    if not df.empty:
        df = df.sort_values(by="date").reset_index(drop=True)

    return df

def convert_itad_to_target_format(df_itad):
    if df_itad.empty:
        return pd.DataFrame(columns=["DateTime", "Final price", "Historical Low", "Retail Price"])

    df = df_itad.copy()

    df["DateTime"] = (
        pd.to_datetime(df["timestamp_raw"], utc=True)
        .dt.tz_convert(None)
        .dt.strftime("%Y-%m-%d %H:%M:%S")
    )

    df=df.rename(columns={"price":"Final price", "regular_price":"Retail Price"})

    df["Historical Low"] = df["Final price"].cummin()

    df=df[["DateTime","Final price","Historical Low", "Retail Price"]]

    return df
    
ITAD_API_KEY = "0c3929a700f88f22c3ae0b877c3fa41fcf3720c7"

def download_game_data(steam_appid):
    return convert_itad_to_target_format(get_game_price_history_itad(steam_appid, ITAD_API_KEY, since_date="2000-01-01T00:00:00Z"))


game_files_list = {
}

def initial_game_processing(df):

    df=df.rename(columns={"Final price":"Current Price","DateTime":"Date"})
    df["Current Price"]=df["Current Price"].replace(0, np.nan)
    df=df.dropna(how="any").reset_index(drop=True)
    df["Date"]=pd.to_datetime(df["Date"])
    df=df[["Date","Current Price","Historical Low","Retail Price"]].sort_values(by=["Date"])
    df_next=df.shift(-1)
    discounts = df[df["Current Price"] < df["Retail Price"]].copy()
    discounts["Days Since Discount"]=discounts["Date"].diff(periods=1).dt.days
    discounts = discounts[
    (discounts["Days Since Discount"] > 0) & 
    (discounts["Days Since Discount"] <= 730)
    ]
    discounts["Month"]=discounts["Date"].dt.month
    discounts["DayofWeek"]=discounts["Date"].dt.dayofweek
    discounts["DayofYear"]=discounts["Date"].dt.dayofyear
    discounts["Occurrence"] = np.ceil(discounts["Date"].dt.day / 7).astype(int)
    discounts["Days Until Next Sale"]=discounts["Days Since Discount"].shift(-1)
    discounts=discounts.dropna(subset=["Days Until Next Sale", "Days Since Discount"])
    return df, discounts


def prediction_function(catalog_dict):
    raw_list = []
    discounts_list = []
    for appid, title in catalog_dict.items():
        raw_game_df = download_game_data(appid)

        if raw_game_df.empty:
            continue

        raw_df, discount_df = initial_game_processing(raw_game_df)

        if (not discount_df.empty) and (len(discount_df) >= 5):
            raw_list.append(raw_df)
            discounts_list.append(discount_df)

    if not discounts_list:
        return "No valid discount data found.", pd.DataFrame()
    rawdata=pd.concat(raw_list, ignore_index=True)
    discounts=pd.concat(discounts_list, ignore_index=True)
    discounts=discounts[discounts["Days Since Discount"]>0]
    
    
    X = discounts.drop(
        columns=["Days Since Discount","Current Price","Days Until Next Sale", "Date", "Last_ATL_Date"], errors="ignore"
    )
    Y = discounts["Days Until Next Sale"]
    
    
    
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestRegressor
    X_train, X_test, Y_train, Y_test, = train_test_split(X,Y,test_size=0.2, random_state=42)
    model = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42)
    model.fit(X_train, Y_train)
    predictions = model.predict(X_test)
    
    
    importance_df = pd.DataFrame(
        {"Feature": X.columns, "Importance": model.feature_importances_}
    ).sort_values("Importance", ascending=False)
    
    
    
    from sklearn.metrics import mean_absolute_error, r2_score
    mae=mean_absolute_error(Y_test,predictions)
    r2=r2_score(Y_test,predictions)
    info=f"Mean Absolute Error: {mae:.2f} days\nR^2 Score: {r2:.2f}"
    comparison=pd.DataFrame({
        "Actual Days Until Next Sale":Y_test.values.ravel(),
        "Predicted Days Until Next Sale":predictions.round(1)
    }).reset_index(drop=True)
    return model, X.columns, importance_df, info, comparison, discounts, X

def create_target_game_features(target_appid, feature_columns):
    raw_game_df = download_game_data(target_appid)
    if raw_game_df.empty:
        return None
    
    retail_price = raw_game_df["Retail Price"].iloc[-1]
    historical_low = raw_game_df["Historical Low"].iloc[-1]
    
    now = pd.Timestamp.now()
    current_features = pd.DataFrame([{
        "Retail Price": retail_price,
        "Historical Low": historical_low,
        "Month": now.month,
        "DayofWeek": now.dayofweek,
        "DayofYear": now.dayofyear,
        "Occurrence": int(np.ceil(now.day / 7))
    }])
    
    return current_features[feature_columns]



def optimize_features_by_importance(X, y, cv_folds=5, min_features=2):

    current_features = list(X.columns)
    
    cv = KFold(n_splits=min(cv_folds, len(y)), shuffle=True, random_state=42)
    
    base_model = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42)
    base_scores = -cross_val_score(base_model, X[current_features], y, cv=cv, scoring="neg_mean_absolute_error")
    best_mae = np.mean(base_scores)
    
    print(f"Initial Baseline MAE ({len(current_features)} features): {best_mae:.2f} days")
    
    while len(current_features) > min_features:
        model = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42)
        model.fit(X[current_features], y)
        
        importances = pd.Series(model.feature_importances_, index=current_features)
        least_important_feature = importances.idxmin()
        candidate_features = [f for f in current_features if f != least_important_feature]
        
        candidate_scores = -cross_val_score(
            RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42),
            X[candidate_features],
            y,
            cv=cv,
            scoring="neg_mean_absolute_error"
        )
        candidate_mae = np.mean(candidate_scores)
        
        if candidate_mae <= best_mae:
            print(f"Dropped '{least_important_feature}' -> New MAE: {candidate_mae:.2f} days (Improved by {best_mae - candidate_mae:.2f} days)")
            best_mae = candidate_mae
            current_features = candidate_features
        else:
            print(f"Stopping: Dropping '{least_important_feature}' worsened MAE ({candidate_mae:.2f} vs {best_mae:.2f} days)")
            break
            
    print(f"\nFinal Optimal Features: {current_features}")
    return current_features, best_mae
















target_appid = "379720"

publisher = get_publisher_by_appid(target_appid)
print(f"Target AppID {target_appid} belongs to publisher: {publisher}")

catalog = publisher_catalog_dict(publisher)

reg_model, feature_cols, importance_df, info, comparison, discounts, X = prediction_function(catalog)

discounts["Is_Sale_Imminent"] = (
    discounts["Days Until Next Sale"] <= 21
).astype(int)

X_clf = discounts[list(X.columns)]
Y_class = discounts["Is_Sale_Imminent"]

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, Y_class, test_size=0.2, random_state=42, stratify=Y_class
)

clf = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
clf.fit(X_train, y_train)

class_preds = clf.predict(X_test)
cm = confusion_matrix(y_test, class_preds)
cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Sale", "Actual: Sale Soon"],
    columns=["Pred: No Sale", "Pred: Sale Soon"],
)
report = classification_report(
    y_test, class_preds, target_names=["No Sale (>21d)", "Sale Soon (<=21d)"]
)

print(info)
print(importance_df)

print(cm_df)
print(report)

target_features = create_target_game_features(target_appid, feature_cols)
if target_features is not None:
    predicted_days = reg_model.predict(target_features)[0]
    sale_imminent = clf.predict(target_features)[0]
    imminent_prob = clf.predict_proba(target_features)[0][1]

    print(f"Game: {catalog.get(str(target_appid), target_appid)}")
    print(f"Predicted Days Until Next Sale: {predicted_days:.1f} days")
    print(f"Sale Imminent within 21 Days: {'Yes' if sale_imminent == 1 else 'No'} ({imminent_prob*100:.1f}% probability)")
else:
    print("Could not retrieve price data for the target game.")

Target AppID 379720 belongs to publisher: Bethesda Softworks
Inspecting 119 potential AppIDs using 6 parallel workers...
Finished processing in 4.09 seconds!

--- Publisher Review Percentile Analysis ---
Total base games evaluated: 67
Calculated 10th percentile cutoff: 262.8 reviews
Remaining games after dropping bottom 10%: 60
Mean Absolute Error: 24.43 days
R^2 Score: 0.25
          Feature  Importance
4       DayofYear    0.907664
3       DayofWeek    0.032117
1    Retail Price    0.030548
0  Historical Low    0.024207
5      Occurrence    0.005464
2           Month    0.000000
                   Pred: No Sale  Pred: Sale Soon
Actual: No Sale              564                1
Actual: Sale Soon             77                5
                   precision    recall  f1-score   support

   No Sale (>21d)       0.88      1.00      0.94       565
Sale Soon (<=21d)       0.83      0.06      0.11        82

         accuracy                           0.88       647
        macro avg       